# Survivor Strategy Analysis
### What separates winners from everyone else — a structured statistical deep-dive

**Pipeline:**
1. Data Loading & Cleaning
2. Key Variable Construction & Validation
3. Era Segmentation
4. Logistic Regression — Univariate → Combined → Per Era
5. Visualizations

---
## 1. Data Loading & Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

df = pd.read_excel(r"C:\Users\sruja\survivor_project\archive\Voting Stats Plus.xlsx")
df.columns = df.columns.str.lower().str.replace(" ", "_")

print(f"Raw shape: {df.shape}")
print(f"Columns : {list(df.columns)}")

In [ ]:
# ── Null audit ────────────────────────────────────────────────────────────────
print("=== Null counts ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string())
print()
print("Notes:")
print("  mergetribecolor  : null = pre-merge elimination. Expected, not a problem.")
print("  startingtribecolor: 4 nulls — cosmetic column, unused in analysis.")
print("  advantagesplayed : null for S37-S48 + a few others. Too many holes to use.")

In [ ]:
# ── Drop players with tribalsattended == 0 ────────────────────────────────────
# These are medical evacuations or boot-before-tribal cases (e.g. Jenna Morasca
# All-Stars, Wanda/Jonathan Palau, Pat Cusack DvG). All ratios would be 0/0.
pre = len(df)
df_work = df[df['tribalsattended'] > 0].copy()
print(f"Dropped {pre - len(df_work)} players with tribalsattended = 0 (medivac / pre-tribal boot).")
print(f"Working rows: {len(df_work)}")

# ── Fix known data entry error ────────────────────────────────────────────────
# Anna Khait, Kaoh Rong: correctlyvoted = 9 but votescast = 1. Impossible.
# She was a pre-merger with 1 vote cast. Corrected to 0.
df_work.loc[df_work['playername'] == 'Anna Khait', 'correctlyvoted'] = 0
print("Fixed: Anna Khait correctlyvoted 9 -> 0 (data entry error).")

---
## 2. Key Variable Construction & Validation

Every variable is a **rate** (normalised by tribals attended or votes cast) so players across different season lengths are directly comparable.

| Variable | Formula | Range | What it measures | Direction for winning |
|---|---|---|---|---|
| `correct_vote_ratio` | correctlyvoted / votescast | [0, 1] | Strategic alignment — voting with the majority | ↑ higher |
| `target_rate` | votesrecieved / tribalsattended | [0, ∞) | How often opponents targeted you | ↓ lower |
| `immunity_rate` | individualimmunites / tribalsattended | [0, 1] | Challenge dominance | ↑ higher |
| `negation_rate` | votesnegated / tribalsattended | [0, ∞) | Idol / advantage usage | ↑ higher |
| `is_winner` | finalplacement == 1 | {0, 1} | **Target variable** | — |

In [ ]:
# ── Build variables ───────────────────────────────────────────────────────────

# correct_vote_ratio: NaN when votescast == 0 (never voted) — not 0
# because 0 votes cast is undefined accuracy, not 0% accuracy
df_work['correct_vote_ratio'] = np.where(
    df_work['votescast'] > 0,
    df_work['correctlyvoted'] / df_work['votescast'],
    np.nan
)
df_work['target_rate']   = df_work['votesrecieved']       / df_work['tribalsattended']
df_work['immunity_rate'] = df_work['individualimmunites']  / df_work['tribalsattended']
df_work['negation_rate'] = df_work['votesnegated']         / df_work['tribalsattended']
df_work['is_winner']     = (df_work['finalplacement'] == 1).astype(int)

KEY_VARS = ['correct_vote_ratio', 'target_rate', 'immunity_rate', 'negation_rate']

# Drop rows where correct_vote_ratio is NaN (player never cast a vote — ~24 rows)
df_clean = df_work.dropna(subset=KEY_VARS).copy()
print(f"Final analysis dataset: {len(df_clean)} rows, {df_clean['is_winner'].sum()} winners")

In [ ]:
# ── Range validation ──────────────────────────────────────────────────────────
print("=== Mathematical range checks ===")
checks = {
    'correct_vote_ratio in [0,1]': df_clean['correct_vote_ratio'].between(0, 1).all(),
    'target_rate >= 0'           : (df_clean['target_rate'] >= 0).all(),
    'immunity_rate in [0,1]'     : df_clean['immunity_rate'].between(0, 1).all(),
    'negation_rate >= 0'         : (df_clean['negation_rate'] >= 0).all(),
    'is_winner in {0,1}'         : df_clean['is_winner'].isin([0,1]).all(),
    'one winner per season'      : (df_clean.groupby('seasonplayed')['is_winner'].sum() == 1).all(),
}
for check, result in checks.items():
    print(f"  {'OK' if result else 'FAIL'}  {check}")

print()
print("=== Descriptive statistics ===")
print(df_clean[KEY_VARS + ['is_winner']].describe().round(3))

In [ ]:
# ── Winner vs Non-Winner means & Pearson correlations ─────────────────────────
print("=== Winner vs Non-Winner means ===")
comp = df_clean.groupby('is_winner')[KEY_VARS].mean().round(3)
comp.index = ['Non-Winner', 'Winner']
print(comp)

print()
print("=== Pearson r with finalplacement (negative = better finish) ===")
for var in KEY_VARS:
    r, p = pearsonr(df_clean[var], df_clean['finalplacement'])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f"  {var:<26}  r = {r:+.3f}  p={p:.2e}  {sig}")

---
## 3. Era Segmentation

In [ ]:
season_to_num = {
    'Borneo': 1, 'Australian Outback': 2, 'Africa': 3, 'Marquesas': 4,
    'Thailand': 5, 'The Amazon': 6, 'Pearl Islands': 7, 'All-Stars': 8,
    'Vanuatu': 9, 'Palau': 10, 'Guatemala': 11, 'Panama': 12,
    'Cook Islands': 13, 'Fiji': 14, 'China': 15, 'Micronesia': 16,
    'Gabon': 17, 'Tocantins': 18, 'Samoa': 19, 'Heroes vs. Villains': 20,
    'Nicaragua': 21, 'Redemption Island': 22, 'South Pacific': 23,
    'Philippines': 24, 'One World': 25, 'Caramoan': 26, 'Blood vs. Water': 27,
    'Cagayan': 28, 'San Juan del Sur': 29, 'Worlds Apart': 30,
    'Cambodia': 31, 'Kaôh Rōng': 32, 'Millenials vs. Gen X': 33,
    'Game Changers': 34, 'Heroes vs. Healers vs. Hustlers': 35,
    'Ghost Island': 36, 'David vs. Goliath': 37, 'Edge of Extinction': 38,
    'Island of the Idols': 39, 'Winners at War': 40,
    'Survivor 41': 41, 'Survivor 42': 42, 'Survivor 43': 43,
    'Survivor 44': 44, 'Survivor 45': 45, 'Survivor 46': 46,
    'Survivor 47': 47, 'Survivor 48': 48,
}

def assign_era(n):
    if pd.isnull(n):  return 'Unknown'
    if n <= 10:       return 'Old School (1-10)'
    if n <= 20:       return 'Middle School (11-20)'
    if n <= 27:       return 'Dark Era (21-27)'
    if n <= 40:       return 'Advantages Era (28-40)'
    return 'New Era (41+)'

df_clean = df_clean.copy()
df_clean['season_num'] = df_clean['seasonplayed'].map(season_to_num)
df_clean['era']        = df_clean['season_num'].apply(assign_era)

ERA_ORDER = [
    'Old School (1-10)', 'Middle School (11-20)', 'Dark Era (21-27)',
    'Advantages Era (28-40)', 'New Era (41+)'
]

unknown = df_clean[df_clean['era'] == 'Unknown']['seasonplayed'].unique()
print('Unmapped seasons:', list(unknown) if len(unknown) else 'None — all mapped')
print()

print(df_clean.groupby('era').agg(
    players=('playername', 'count'),
    seasons=('seasonplayed', 'nunique'),
    winners=('is_winner', 'sum')
).reindex(ERA_ORDER))

In [ ]:
# ── Advantages Era subset ─────────────────────────────────────────────────────
df_adv = df_clean[df_clean['era'] == 'Advantages Era (28-40)'].copy()
print(f"Advantages Era (S28-S40): {len(df_adv)} players, {df_adv['is_winner'].sum()} winners")
print()
print("Winner vs Non-Winner means — Advantages Era:")
comp_adv = df_adv.groupby('is_winner')[KEY_VARS].mean().round(3)
comp_adv.index = ['Non-Winner', 'Winner']
print(comp_adv)

---
## 4. Logistic Regression

**Why logistic regression?**  
`is_winner` is binary (0/1). Logistic regression gives:
- **AUC** — discriminative power (0.5 = random, 1.0 = perfect)
- **Standardised coefficients** — directly comparable importance weights (features z-scored first)
- **Sign** — confirms direction aligns with domain expectations

`class_weight='balanced'` is essential because winners are only ~6% of rows. Without it, the model predicts 'non-winner' for everyone and achieves 94% accuracy while being useless.

`StratifiedKFold` ensures each fold has proportional winners — critical with only 13 winners in the Advantages Era.

In [ ]:
def run_logistic(X_df, y, label=''):
    scaler = StandardScaler()
    X_s    = scaler.fit_transform(X_df)
    lr     = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
    skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs   = cross_val_score(lr, X_s, y, cv=skf, scoring='roc_auc')
    lr.fit(X_s, y)
    return {
        'label'   : label,
        'auc_mean': aucs.mean(),
        'auc_std' : aucs.std(),
        'coefs'   : dict(zip(X_df.columns, lr.coef_[0])),
        'features': list(X_df.columns),
    }

In [ ]:
# ── 4a. Univariate models — full dataset ──────────────────────────────────────
# One variable at a time: which single metric best discriminates winners?
print("=== Univariate Logistic Regression — Full Dataset ===")
print(f"{'Variable':<26} {'AUC':>6}  {'+-':>5}  {'Coef':>7}  Direction")
print("-" * 68)

uni_results = {}
y_all = df_clean['is_winner']

for var in KEY_VARS:
    res  = run_logistic(df_clean[[var]], y_all, label=var)
    uni_results[var] = res
    coef = res['coefs'][var]
    print(f"{var:<26} {res['auc_mean']:.3f}  +-{res['auc_std']:.3f}  {coef:+.3f}   "
          f"{'more wins' if coef > 0 else 'less wins'}")

best = max(uni_results, key=lambda v: uni_results[v]['auc_mean'])
print(f"\n=> Single strongest predictor: {best}  (AUC {uni_results[best]['auc_mean']:.3f})")

In [ ]:
# ── 4b. Combined model — full dataset ─────────────────────────────────────────
combined_all = run_logistic(df_clean[KEY_VARS], y_all, label='Combined')

print("=== Combined Model — Full Dataset ===")
print(f"ROC-AUC: {combined_all['auc_mean']:.3f} +- {combined_all['auc_std']:.3f}")
print()
print("Feature importance (standardised coefficients, ranked by magnitude):")
for feat, coef in sorted(combined_all['coefs'].items(), key=lambda x: abs(x[1]), reverse=True):
    bar = 'X' * int(abs(coef) * 5)
    print(f"  {feat:<26} {coef:+.3f}  {bar}")

In [ ]:
# ── 4c. Univariate + combined — Advantages Era ───────────────────────────────
y_adv = df_adv['is_winner']

print("=== Univariate — Advantages Era ===")
print(f"{'Variable':<26} {'AUC':>6}  {'+-':>5}  {'Coef':>7}")
print("-" * 52)
uni_adv = {}
for var in KEY_VARS:
    res = run_logistic(df_adv[[var]], y_adv, label=var)
    uni_adv[var] = res
    print(f"{var:<26} {res['auc_mean']:.3f}  +-{res['auc_std']:.3f}  {res['coefs'][var]:+.3f}")

print()
combined_adv = run_logistic(df_adv[KEY_VARS], y_adv, label='Combined')
print(f"=== Combined — Advantages Era ===")
print(f"ROC-AUC: {combined_adv['auc_mean']:.3f} +- {combined_adv['auc_std']:.3f}")
print()
for feat, coef in sorted(combined_adv['coefs'].items(), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:<26} {coef:+.3f}")

In [ ]:
# ── 4d. Era-by-era combined models ────────────────────────────────────────────
print(f"{'Era':<26} {'AUC':>6}  {'+-':>5}  | "
      f"{'cvr':>6}  {'tgt':>6}  {'imm':>6}  {'neg':>6}")
print("-" * 74)

era_results = {}
for era in ERA_ORDER:
    sub = df_clean[df_clean['era'] == era]
    if sub['is_winner'].sum() < 3:
        print(f"{era:<26}  -- insufficient winners")
        continue
    res = run_logistic(sub[KEY_VARS], sub['is_winner'], label=era)
    era_results[era] = res
    c = res['coefs']
    print(f"{era:<26} {res['auc_mean']:.3f}  +-{res['auc_std']:.3f}  | "
          f"{c['correct_vote_ratio']:+.2f}  "
          f"{c['target_rate']:+.2f}  "
          f"{c['immunity_rate']:+.2f}  "
          f"{c['negation_rate']:+.2f}")

---
## 5. Visualizations

In [ ]:
BLUE  = '#1565C0'
LBLUE = '#90CAF9'
RED   = '#EF5350'
DBLUE = '#0D47A1'
DRED  = '#B71C1C'

titles = {
    'correct_vote_ratio': 'Correct Vote Ratio  (higher = better)',
    'target_rate'       : 'Target Rate  (lower = better)',
    'immunity_rate'     : 'Immunity Rate  (higher = better)',
    'negation_rate'     : 'Negation Rate / Idols  (higher = better)',
}

In [ ]:
# ── Plot 1: Univariate AUC ranking (full dataset) ─────────────────────────────
vars_sorted = sorted(uni_results.items(), key=lambda x: x[1]['auc_mean'], reverse=True)
labels_u = [v for v, _ in vars_sorted]
aucs_u   = [r['auc_mean'] for _, r in vars_sorted]
stds_u   = [r['auc_std']  for _, r in vars_sorted]
colors_u = [BLUE if a == max(aucs_u) else LBLUE for a in aucs_u]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels_u, aucs_u, xerr=stds_u, color=colors_u,
               capsize=4, edgecolor='white', height=0.55)
ax.axvline(0.5, color='grey', linestyle='--', linewidth=1, label='Random baseline (AUC=0.5)')
for bar, auc in zip(bars, aucs_u):
    ax.text(auc + 0.008, bar.get_y() + bar.get_height()/2,
            f'{auc:.3f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('ROC-AUC (5-fold stratified CV)')
ax.set_title('Which single variable best predicts winning?', fontweight='bold')
ax.set_xlim(0.4, 1.05)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Winner vs Non-Winner distributions — Advantages Era ───────────────
winners_adv = df_adv[df_adv['is_winner'] == 1]
losers_adv  = df_adv[df_adv['is_winner'] == 0]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, var in zip(axes.flatten(), KEY_VARS):
    ax.hist(losers_adv[var],  bins=20, alpha=0.55, color=RED,  label='Non-Winners', density=True)
    ax.hist(winners_adv[var], bins=20, alpha=0.80, color=BLUE, label='Winners',     density=True)
    wm = winners_adv[var].mean()
    lm = losers_adv[var].mean()
    ax.axvline(wm, color=DBLUE, linestyle='--', linewidth=2, label=f'Winner mean: {wm:.2f}')
    ax.axvline(lm, color=DRED,  linestyle='--', linewidth=2, label=f'Non-winner mean: {lm:.2f}')
    ax.set_title(titles[var], fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle('Advantages Era — Variable Distributions: Winners vs Non-Winners',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: AUC across eras ───────────────────────────────────────────────────
era_labels = [e for e in ERA_ORDER if e in era_results]
era_aucs   = [era_results[e]['auc_mean'] for e in era_labels]
era_stds   = [era_results[e]['auc_std']  for e in era_labels]

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(era_labels,
                [a - s for a, s in zip(era_aucs, era_stds)],
                [a + s for a, s in zip(era_aucs, era_stds)],
                alpha=0.15, color=BLUE)
ax.plot(era_labels, era_aucs, 'o-', color=BLUE, linewidth=2.5,
        markersize=9, markerfacecolor='white', markeredgewidth=2.5)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1)
for x, auc in zip(era_labels, era_aucs):
    ax.annotate(f'{auc:.3f}', (x, auc), textcoords='offset points',
                xytext=(0, 12), ha='center', fontsize=9, fontweight='bold')
ax.set_ylim(0.4, 1.1)
ax.set_ylabel('ROC-AUC (5-fold CV)')
ax.set_title('How well can we predict winners across eras?', fontweight='bold')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 4: Coefficient heatmap across eras ───────────────────────────────────
coef_df = pd.DataFrame(
    {era: era_results[era]['coefs'] for era in era_labels}
).T[KEY_VARS]

fig, ax = plt.subplots(figsize=(11, 4))
vmax = max(abs(coef_df.values.min()), abs(coef_df.values.max()))
im = ax.imshow(coef_df.values, cmap='RdBu', aspect='auto', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(KEY_VARS)))
ax.set_xticklabels(KEY_VARS, rotation=20, ha='right')
ax.set_yticks(range(len(era_labels)))
ax.set_yticklabels(era_labels)
for i in range(len(era_labels)):
    for j in range(len(KEY_VARS)):
        val = coef_df.iloc[i, j]
        ax.text(j, i, f'{val:+.2f}', ha='center', va='center', fontsize=10,
                color='white' if abs(val) > vmax * 0.55 else 'black')
plt.colorbar(im, ax=ax, label='Standardised coefficient')
ax.set_title('How variable importance shifts across eras\n'
             '(Blue = associated with winning, Red = associated with losing)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 5: Winner vs Non-Winner averages per era (grouped bars) ──────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
x = np.arange(len(ERA_ORDER))
short_labels = ['Old\nSchool', 'Middle\nSchool', 'Dark\nEra',
                'Advantages\nEra', 'New\nEra']

for ax, var in zip(axes.flatten(), KEY_VARS):
    w_vals = df_clean[df_clean['is_winner']==1].groupby('era')[var].mean().reindex(ERA_ORDER)
    l_vals = df_clean[df_clean['is_winner']==0].groupby('era')[var].mean().reindex(ERA_ORDER)
    ax.bar(x - 0.22, w_vals, 0.4, label='Winners',     color=BLUE, alpha=0.85)
    ax.bar(x + 0.22, l_vals, 0.4, label='Non-Winners', color=RED,  alpha=0.70)
    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, fontsize=9)
    ax.set_title(titles[var], fontweight='bold')
    ax.legend(fontsize=8)

fig.suptitle('Winner vs Non-Winner Averages Across All Eras',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 6: Strategy quadrant scatter — Advantages Era ────────────────────────
# The two strongest predictors on one plot.
# Ideal winners cluster bottom-right: high correct_vote_ratio, low target_rate.
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(df_adv[df_adv['is_winner']==0]['correct_vote_ratio'],
           df_adv[df_adv['is_winner']==0]['target_rate'],
           c=RED,  s=30,  alpha=0.55, label='Non-Winner', zorder=2)
ax.scatter(df_adv[df_adv['is_winner']==1]['correct_vote_ratio'],
           df_adv[df_adv['is_winner']==1]['target_rate'],
           c=BLUE, s=130, alpha=0.90, label='Winner', marker='*', zorder=3)

xm = df_adv['correct_vote_ratio'].median()
ym = df_adv['target_rate'].median()
ax.axvline(xm, color='grey', linestyle=':', linewidth=0.9)
ax.axhline(ym, color='grey', linestyle=':', linewidth=0.9)

fs = 8
ax.text(0.02, 0.98, 'Misaligned\n+ Targeted',   transform=ax.transAxes, va='top',    fontsize=fs, color=DRED)
ax.text(0.68, 0.98, 'Aligned\n+ Targeted',       transform=ax.transAxes, va='top',    fontsize=fs, color='#6A1B9A')
ax.text(0.02, 0.02, 'Misaligned\n+ Safe',         transform=ax.transAxes, va='bottom', fontsize=fs, color='#E65100')
ax.text(0.68, 0.02, 'Aligned + Safe\n<- IDEAL',   transform=ax.transAxes, va='bottom', fontsize=fs, color=DBLUE, fontweight='bold')

ax.set_xlabel('Correct Vote Ratio  ->  strategic alignment', fontsize=11)
ax.set_ylabel('Target Rate  ->  threat perceived by others', fontsize=11)
ax.set_title('Strategic Alignment vs Threat Level — Advantages Era\n'
             'Winners cluster bottom-right: aligned + avoided', fontweight='bold')
ax.legend(fontsize=11, markerscale=1.2)
plt.tight_layout()
plt.show()